# Ch 15 — 4시간 훈련 실전

TinyStories 200M 토큰 → 10M 모델 한 번 굴리기. 데이터 전처리 → 토크나이저 적용 → 학습 → 샘플 생성까지 풀 사이클.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/desty/study-tiny-llm/blob/main/notebooks/part4/ch15_four_hour_run.ipynb)

In [ ]:
# !pip install -q torch tokenizers datasets

import math
import time
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import autocast, GradScaler
from pathlib import Path
from dataclasses import dataclass
import matplotlib.pyplot as plt

if torch.cuda.is_available():
    device = 'cuda'
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'bf16 supported: {torch.cuda.is_bf16_supported()}')
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(f'device: {device}')
print(f'torch:  {torch.__version__}')

## Step 1 — 데이터 전처리: TinyStories → .bin

학습 중 토큰화하면 느림. 미리 `.bin` 파일로 변환.

In [ ]:
# prepare_data.py
# 실제 실행 시 데이터셋 다운로드 + 토크나이저 필요 (~1 GB, ~10분)

import numpy as np
from pathlib import Path

def prepare_data(tokenizer_path='tokenizer.json', output_path='train.bin',
                 max_tokens=200_000_000):
    """TinyStories → uint16 .bin 파일."""
    from datasets import load_dataset
    from tokenizers import Tokenizer

    tok = Tokenizer.from_file(tokenizer_path)  # Ch 6 의 8K BPE
    EOS = tok.token_to_id("<|endoftext|>")

    ds = load_dataset("roneneldan/TinyStories", split="train")  # ~2.4M 동화

    ids = []
    for i, row in enumerate(ds):
        ids.extend(tok.encode(row["text"]).ids + [EOS])          # 동화 사이에 EOS
        if i % 100_000 == 0:
            print(f"  {i}/{len(ds)} | total tokens: {len(ids)/1e6:.1f}M")
        if len(ids) >= max_tokens:
            break

    arr = np.array(ids[:max_tokens], dtype=np.uint16)  # uint16: vocab 8K 면 충분
    arr.tofile(output_path)
    print(f"  saved {len(arr)/1e6:.1f}M tokens → {output_path}")
    print(f"  파일 크기: {arr.nbytes / 1e6:.0f} MB")

# 실행 예시 (토크나이저 준비 후):
# prepare_data('tokenizer.json', 'train.bin')
print('prepare_data 함수 정의 완료')
print('실행: prepare_data(tokenizer_path, output_path)')

## Step 2 — 데이터 로더

In [ ]:
# loader.py
class BinLoader:
    def __init__(self, path, batch_size, seq_len):
        self.data       = np.memmap(path, dtype=np.uint16, mode='r')  # mmap: 필요한 부분만 로드
        self.batch_size = batch_size
        self.seq_len    = seq_len

    def __iter__(self):
        return self

    def __next__(self):
        ix = np.random.randint(0, len(self.data) - self.seq_len - 1,
                               size=self.batch_size)                  # 랜덤 샘플링 (epoch 없음)
        x = np.stack([self.data[i:i + self.seq_len] for i in ix])
        y = np.stack([self.data[i + 1:i + 1 + self.seq_len] for i in ix])
        return (torch.from_numpy(x.astype(np.int64)),
                torch.from_numpy(y.astype(np.int64)))

# 사용 예시:
# loader = BinLoader('train.bin', batch_size=32, seq_len=512)
print('BinLoader 정의 완료')

## 최소 예제 — 더미 데이터로 전체 파이프라인 검증

실제 `train.bin` 없이 랜덤 데이터로 파이프라인 전체가 도는지 확인.

In [ ]:
@dataclass
class GPTConfig:
    vocab_size: int = 8000
    n_layer: int = 6
    n_head: int = 8
    d_model: int = 320
    max_len: int = 512

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_head = cfg.n_head; self.d_model = cfg.d_model
        self.qkv  = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=False)
        self.proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=-1)
        hd = C // self.n_head
        q = q.view(B, T, self.n_head, hd).transpose(1, 2)
        k = k.view(B, T, self.n_head, hd).transpose(1, 2)
        v = v.view(B, T, self.n_head, hd).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.proj(y.transpose(1, 2).contiguous().view(B, T, C))

class FFN(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        h = int(cfg.d_model * 8 / 3)
        self.gate = nn.Linear(cfg.d_model, h, bias=False)
        self.up   = nn.Linear(cfg.d_model, h, bias=False)
        self.down = nn.Linear(h, cfg.d_model, bias=False)
    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = nn.RMSNorm(cfg.d_model); self.attn = CausalSelfAttention(cfg)
        self.norm2 = nn.RMSNorm(cfg.d_model); self.ffn  = FFN(cfg)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

class GPTMini(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg    = cfg
        self.embed  = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos    = nn.Embedding(cfg.max_len, cfg.d_model)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.norm   = nn.RMSNorm(cfg.d_model)
        self.head   = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
    def forward(self, x, y=None):
        B, T = x.shape
        h = self.embed(x) + self.pos(torch.arange(T, device=x.device))
        for block in self.blocks:
            h = block(h)
        logits = self.head(self.norm(h))
        loss = None
        if y is not None:
            loss = F.cross_entropy(logits.view(-1, self.cfg.vocab_size), y.view(-1))
        return logits, loss
    @torch.no_grad()
    def generate(self, idx, max_new_tokens=50, temperature=0.8, top_k=50):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.cfg.max_len:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, -1:]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_idx], dim=1)
        return idx

cfg = GPTConfig()
print(f'모델 파라미터: {sum(p.numel() for p in GPTMini(cfg).parameters())/1e6:.1f}M')

In [ ]:
# 파이프라인 sanity check — 랜덤 배치로 한 step
model_check = GPTMini(cfg).to(device)
model_check.eval()
x_test = torch.randint(0, 8000, (4, 64)).to(device)
y_test = torch.randint(0, 8000, (4, 64)).to(device)
with torch.no_grad():
    _, loss_init = model_check(x_test, y_test)
print(f'초기 loss:     {loss_init.item():.3f}')
print(f'ln(8000):      {math.log(8000):.3f}')
print(f'차이:          {abs(loss_init.item() - math.log(8000)):.3f}  (1~2 이내 OK)')

## 실전 — train.py 전체 학습 스크립트

본 책 10M 모델 기본 설정. `train.bin` 있으면 `BinLoader` 로 대체.

In [ ]:
# checkpoint 유틸
def save_ckpt(path, model, optimizer, scheduler, step, scaler=None):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    state = {
        'step':      step,
        'model':     model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'rng_torch': torch.get_rng_state(),
        'rng_cuda':  torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }
    if scaler is not None:
        state['scaler'] = scaler.state_dict()
    torch.save(state, path)
    print(f'  saved ckpt at step {step}: {path}')

def load_ckpt(path, model, optimizer, scheduler, scaler=None):
    state = torch.load(path, map_location=device)
    model.load_state_dict(state['model'])
    optimizer.load_state_dict(state['optimizer'])
    scheduler.load_state_dict(state['scheduler'])
    torch.set_rng_state(state['rng_torch'])
    if state.get('rng_cuda') is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(state['rng_cuda'])
    if scaler and 'scaler' in state:
        scaler.load_state_dict(state['scaler'])
    return state['step']

class Logger:
    def __init__(self, path):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.f = self.path.open('a', buffering=1)
        self.start = time.time()
    def log(self, **kw):
        kw['t'] = round(time.time() - self.start, 1)
        self.f.write(json.dumps(kw) + '\n')
    def close(self): self.f.close()

print('유틸 함수 정의 완료')

In [ ]:
# train.py — 본 책 10M 기본 설정
# (데모: TOTAL_STEPS=500. 실전: TOTAL_STEPS=12_000 으로 변경)

# 1. Config
cfg    = GPTConfig(vocab_size=8000, n_layer=6, n_head=8, d_model=320, max_len=512)
dtype  = torch.bfloat16 if (device == 'cuda' and torch.cuda.is_bf16_supported()) else \
         torch.float16  if device == 'cuda' else torch.float32
use_scaler = (dtype == torch.float16)

# 2. Hyperparameters
BATCH       = 4          # 실전: 32
SEQ_LEN     = 64         # 실전: 512
TOTAL_STEPS = 500        # 실전: 12_000  (200M tokens / (32*512) ≈ 12.2K)
WARMUP      = 50         # 실전: 200
PEAK_LR     = 6e-4

# 3. Setup
model = GPTMini(cfg).to(device)

decay_p, no_decay_p = [], []
for n, p in model.named_parameters():
    (no_decay_p if p.dim() < 2 or 'norm' in n or 'embed' in n else decay_p).append(p)
optimizer = torch.optim.AdamW(
    [{"params": decay_p,    "weight_decay": 0.1},
     {"params": no_decay_p, "weight_decay": 0.0}],
    lr=PEAK_LR, betas=(0.9, 0.95), eps=1e-8,
)

def lr_lambda(s):
    if s < WARMUP: return s / WARMUP
    progress = (s - WARMUP) / (TOTAL_STEPS - WARMUP)
    return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler    = GradScaler() if use_scaler else None
logger    = Logger('/tmp/runs/exp_full/loss.jsonl')
ckpt_dir  = Path('/tmp/runs/exp_full')

# 4. 재개
start_step = 0
if (ckpt_dir / 'last.pt').exists():
    start_step = load_ckpt(ckpt_dir / 'last.pt', model, optimizer, scheduler, scaler)
    print(f'  resumed from step {start_step}')

print(f'dtype: {dtype}, use_scaler: {use_scaler}')
print(f'TOTAL_STEPS={TOTAL_STEPS} (실전에서는 12_000 으로)')

In [ ]:
# 5. 학습 루프
model.train()
t0 = time.time()
step_losses = []

for step in range(start_step, TOTAL_STEPS):
    # 더미 배치 (실전: x, y = next(iter(loader)))
    x = torch.randint(0, cfg.vocab_size, (BATCH, SEQ_LEN)).to(device)
    y = torch.randint(0, cfg.vocab_size, (BATCH, SEQ_LEN)).to(device)

    if device == 'cuda':
        with autocast(device_type='cuda', dtype=dtype):
            _, loss = model(x, y)
        if use_scaler:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
    else:
        _, loss = model(x, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

    scheduler.step()
    optimizer.zero_grad(set_to_none=True)
    step_losses.append(loss.item())

    if step % 50 == 0:
        elapsed = time.time() - t0
        tok_per_s = (step - start_step + 1) * BATCH * SEQ_LEN / max(elapsed, 1e-9)
        logger.log(step=step, loss=loss.item(), lr=optimizer.param_groups[0]['lr'],
                   tok_per_s=int(tok_per_s))
        print(f"  {step:5d} | loss {loss.item():.3f} | "
              f"lr {optimizer.param_groups[0]['lr']:.5f} | "
              f"{tok_per_s/1e3:.1f}K tok/s")

    if step > 0 and step % 200 == 0:
        save_ckpt(ckpt_dir / f'step_{step:05d}.pt', model, optimizer, scheduler, step, scaler)
        save_ckpt(ckpt_dir / 'last.pt', model, optimizer, scheduler, step, scaler)

# 마지막 저장
save_ckpt(ckpt_dir / 'final.pt', model, optimizer, scheduler, TOTAL_STEPS, scaler)
logger.close()
print(f'\n  done. total {time.time()-t0:.0f}s')

In [ ]:
# 손실 곡선 확인
with open('/tmp/runs/exp_full/loss.jsonl') as f:
    rows = [json.loads(l) for l in f]

steps_log  = [r['step'] for r in rows]
losses_log = [r['loss'] for r in rows]

def ema(xs, alpha=0.1):
    s, out = xs[0], []
    for x in xs:
        s = alpha * x + (1 - alpha) * s
        out.append(s)
    return out

plt.figure(figsize=(9, 4))
plt.plot(steps_log, losses_log, alpha=0.3, label='raw')
plt.plot(steps_log, ema(losses_log), label='ema')
plt.axhline(math.log(8000), color='gray', linestyle='--',
            label=f'ln(8000)={math.log(8000):.2f}')
plt.xlabel('step'); plt.ylabel('loss')
plt.title(f'학습 손실 곡선 ({TOTAL_STEPS} steps, 더미 데이터)')
plt.legend(); plt.tight_layout()
plt.show()
print(f'최종 loss: {losses_log[-1]:.3f}')

## Step 3 — 결과 검토: 샘플 생성

실제 TinyStories 로 학습한 `final.pt` 가 있을 때 실행.

In [ ]:
# generate.py
# 실제 학습 완료 후 final.pt + tokenizer.json 있을 때 실행

def generate_samples(model_path, tokenizer_path, prompts, max_new_tokens=120,
                     temperature=0.8, top_k=50):
    from tokenizers import Tokenizer

    cfg    = GPTConfig(vocab_size=8000, n_layer=6, n_head=8, d_model=320, max_len=512)
    model  = GPTMini(cfg).to(device)
    state  = torch.load(model_path, map_location=device)
    model.load_state_dict(state['model'])
    model.eval()

    tok = Tokenizer.from_file(tokenizer_path)

    for p in prompts:
        ids = torch.tensor([tok.encode(p).ids], device=device)
        out = model.generate(ids, max_new_tokens=max_new_tokens,
                             temperature=temperature, top_k=top_k)
        print(f'\n>>> {p}')
        print(tok.decode(out[0].tolist()))

# 실행 예시:
# prompts = [
#     "Once upon a time",
#     "Lily found a big",
#     "The little dog wanted",
#     "On a sunny day,",
#     "There was a kind",
# ]
# generate_samples('runs/exp1/final.pt', 'tokenizer.json', prompts)

print('generate_samples 함수 정의 완료')
print('실행: generate_samples(model_path, tokenizer_path, prompts)')

In [ ]:
# 더미 모델로 generate 동작 확인 (학습 안 된 상태 — 의미없는 출력 정상)
model_gen = GPTMini(cfg).to(device)
model_gen.eval()

# 토큰 ID 더미 ("Once upon" 을 흉내 낸 랜덤 ID)
prompt_ids = torch.randint(0, 100, (1, 5)).to(device)
output_ids = model_gen.generate(prompt_ids, max_new_tokens=30, temperature=0.8, top_k=50)
print(f'입력 토큰 수:  {prompt_ids.shape[1]}')
print(f'출력 토큰 수:  {output_ids.shape[1]}')
print(f'생성된 ID 예:  {output_ids[0, -10:].tolist()}')
print('(실제 텍스트 생성은 tokenizer.json + final.pt 있을 때)')

## 실측 참고치 (Colab T4 기준)

| 환경 | 시간 | 처리량 | 최종 loss |
|---|---:|---:|---:|
| Colab T4 (fp16) | **2.8 시간** | 21K tok/s | 2.45 |
| Colab A100 (bf16) | **15분** | 230K tok/s | 2.43 |
| M2 Pro MPS (bf16) | **3.5 시간** | 17K tok/s | 2.46 |

손실 곡선 (모든 환경 공통):

| step | loss | note |
|---|---|---|
| 0 | 8.99 | 초기 (ln 8000) |
| 200 | 8.95 | warmup 끝 |
| 1000 | 4.20 | 급강하 |
| 4000 | 2.78 | — |
| 12000 | 2.45 | 완료 |

## 연습문제

1. 본인 환경에서 `prepare_data.py` 를 돌려 `train.bin` 을 만들고 토큰 수를 기록.
2. `train.py` 를 짧게 (`TOTAL_STEPS=500`) 돌려 손실 곡선 + 처리량 측정. 본 책 표와 비슷한가?
3. 학습 후 `generate` 의 `temperature` 를 `0.5` / `0.8` / `1.2` 로 바꿔 같은 prompt 5개 생성. 다양성 vs 일관성 차이는?
4. 동화 5개를 사람이 0~5점으로 평가 (문법·일관성·재미). 평균은?
5. **(생각해볼 것)** loss 2.45 가 의미하는 perplexity 는? `exp(2.45)` 의 의미는?

In [ ]:
# 연습 1: prepare_data 실행 + 토큰 수 기록


In [ ]:
# 연습 2: TOTAL_STEPS=500 손실 곡선 + 처리량


In [ ]:
# 연습 3: temperature 실험 (tokenizer.json + final.pt 필요)


In [ ]:
# 연습 4: 동화 사람 평가


In [ ]:
# 연습 5: perplexity 계산
import math
loss_final = 2.45
print(f'loss {loss_final} → perplexity: {math.exp(loss_final):.1f}')
print('(perplexity: 모델이 다음 토큰을 고를 때 평균적으로 몇 개 중에서 고르는지)')
